Démarrage de Spark

In [19]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_CONF_DIR"] = os.path.expanduser("~/hadoop/etc/hadoop")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Merge Google Form Responses")
    .master("local[*]")
    .getOrCreate()
)

spark

Lecture des deux CSV

In [20]:
base_path = "hdfs://localhost:9000/projet/gaming_mental_health/input/gaming_mental_health.csv"
form_path = "file://" + os.path.expanduser("~/projets/gaming_mental_health/gform/form_responses.csv")

base_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path)
)

form_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(form_path)
)

print("Colonnes dataset original :")
print(base_df.columns)

print("\nColonnes Google Form :")
print(form_raw_df.columns)

print("\nNombre de lignes dataset original :", base_df.count())
print("Nombre de réponses Google Form :", form_raw_df.count())

Colonnes dataset original :
['record_id', 'age', 'gender', 'daily_gaming_hours', 'game_genre', 'primary_game', 'gaming_platform', 'sleep_hours', 'sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'grades_gpa', 'work_productivity_score', 'mood_state', 'mood_swing_frequency', 'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems', 'eye_strain', 'back_neck_pain', 'weight_change_kg', 'exercise_hours_weekly', 'social_isolation_score', 'face_to_face_social_hours_weekly', 'monthly_game_spending_usd', 'years_gaming', 'gaming_addiction_risk_level']

Colonnes Google Form :
['Horodateur', 'Age', 'Gender', 'Average daily gaming time', 'Main game genre', 'Main game played', 'Main gaming platform', 'For how many years have you been playing video games?', 'How much money do you spend on video games per month on average?', 'How many hours do you sleep per night on average?', 'Sleep quality ', 'How often is your sleep disrupted?', 'What is your approxima

Renommage des colonnes du gform pour correspondre au csv

In [21]:
from pyspark.sql.functions import col

column_mapping = {
    "Age": "age",
    "Gender": "gender",
    "Average daily gaming time": "daily_gaming_hours",
    "Main game genre": "game_genre",
    "Main game played": "primary_game",
    "Main gaming platform": "gaming_platform",
    "For how many years have you been playing video games?": "years_gaming",
    "How much money do you spend on video games per month on average?": "monthly_game_spending_usd",
    "How many hours do you sleep per night on average?": "sleep_hours",
    "Sleep quality ": "sleep_quality",
    "How often is your sleep disrupted?": "sleep_disruption_frequency",
    "What is your approximate average grade out of 20?": "grade_out_of_20",
    "How would you rate your productivity in studies or work?": "work_productivity_score",
    "Which mood best describes you recently?": "mood_state",
    "How often do you experience mood swings?": "mood_swing_frequency",
    "Do you feel frustration, stress, or discomfort when you cannot play video games?": "withdrawal_symptoms",
    "Have you lost interest in other activities because of gaming?": "loss_of_other_interests",
    "Do you continue playing video games despite problems caused by gaming?": "continued_despite_problems",
    "Do you often experience eye strain related to gaming?": "eye_strain",
    "Do you often experience back or neck pain related to gaming?": "back_neck_pain",
    "What is your approximate recent weight change in kilograms?": "weight_change_kg",
    "How many hours do you exercise per week on average?": "exercise_hours_weekly",
    "How socially isolated do you feel?": "social_isolation_score",
    "How many hours do you spend socializing face-to-face per week?": "face_to_face_social_hours_weekly",
}

form_df = form_raw_df

if "Horodateur" in form_df.columns:
    form_df = form_df.drop("Horodateur")

if "Timestamp" in form_df.columns:
    form_df = form_df.drop("Timestamp")

for old_name, new_name in column_mapping.items():
    if old_name in form_df.columns:
        form_df = form_df.withColumnRenamed(old_name, new_name)

form_df.printSchema()
form_df.show(5)

root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- daily_gaming_hours: double (nullable = true)
 |-- game_genre: string (nullable = true)
 |-- primary_game: string (nullable = true)
 |-- gaming_platform: string (nullable = true)
 |-- years_gaming: double (nullable = true)
 |-- monthly_game_spending_usd: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- sleep_quality: string (nullable = true)
 |-- sleep_disruption_frequency: string (nullable = true)
 |-- grade_out_of_20: integer (nullable = true)
 |-- work_productivity_score: integer (nullable = true)
 |-- mood_state: string (nullable = true)
 |-- mood_swing_frequency: string (nullable = true)
 |-- withdrawal_symptoms: string (nullable = true)
 |-- loss_of_other_interests: string (nullable = true)
 |-- continued_despite_problems: string (nullable = true)
 |-- eye_strain: string (nullable = true)
 |-- back_neck_pain: string (nullable = true)
 |-- weight_change_kg: double (nullable =

Convertion des types et valeurs

In [22]:
from pyspark.sql.functions import when, round as spark_round

# Colonnes Yes / No -> True / False
boolean_columns = [
    "withdrawal_symptoms",
    "loss_of_other_interests",
    "continued_despite_problems",
    "eye_strain",
    "back_neck_pain",
]

for c in boolean_columns:
    form_df = form_df.withColumn(
        c,
        when(col(c) == "Yes", True)
        .when(col(c) == "No", False)
        .otherwise(col(c))
    )

# Colonnes numériques
numeric_columns = [
    "age",
    "daily_gaming_hours",
    "years_gaming",
    "monthly_game_spending_usd",
    "sleep_hours",
    "grade_out_of_20",
    "work_productivity_score",
    "weight_change_kg",
    "exercise_hours_weekly",
    "social_isolation_score",
    "face_to_face_social_hours_weekly",
]

for c in numeric_columns:
    form_df = form_df.withColumn(c, col(c).cast("double"))

# Conversion note /20 -> GPA /4
form_df = form_df.withColumn(
    "grades_gpa",
    spark_round(col("grade_out_of_20") / 5, 2)
)

form_df = form_df.drop("grade_out_of_20")

form_df.printSchema()
form_df.show(5)

root
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- daily_gaming_hours: double (nullable = true)
 |-- game_genre: string (nullable = true)
 |-- primary_game: string (nullable = true)
 |-- gaming_platform: string (nullable = true)
 |-- years_gaming: double (nullable = true)
 |-- monthly_game_spending_usd: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- sleep_quality: string (nullable = true)
 |-- sleep_disruption_frequency: string (nullable = true)
 |-- work_productivity_score: double (nullable = true)
 |-- mood_state: string (nullable = true)
 |-- mood_swing_frequency: string (nullable = true)
 |-- withdrawal_symptoms: boolean (nullable = true)
 |-- loss_of_other_interests: boolean (nullable = true)
 |-- continued_despite_problems: boolean (nullable = true)
 |-- eye_strain: boolean (nullable = true)
 |-- back_neck_pain: boolean (nullable = true)
 |-- weight_change_kg: double (nullable = true)
 |-- exercise_hours_weekly: double (nu

Génération d'une colonne pas présente dans le gform mais présente dans le csv

In [23]:
form_df = form_df.withColumn(
    "academic_work_performance",
    when((col("grades_gpa") >= 3.4) & (col("work_productivity_score") >= 8), "Excellent")
    .when((col("grades_gpa") >= 2.8) & (col("work_productivity_score") >= 6), "Good")
    .when((col("grades_gpa") >= 2.0) & (col("work_productivity_score") >= 4), "Average")
    .when((col("grades_gpa") >= 1.2) | (col("work_productivity_score") >= 3), "Below Average")
    .otherwise("Poor")
)

form_df.select(
    "grades_gpa",
    "work_productivity_score",
    "academic_work_performance"
).show()

+----------+-----------------------+-------------------------+
|grades_gpa|work_productivity_score|academic_work_performance|
+----------+-----------------------+-------------------------+
|       2.2|                    4.0|                  Average|
|       3.0|                    3.0|            Below Average|
|       2.4|                    6.0|                  Average|
|       3.0|                    8.0|                     Good|
|       2.6|                    4.0|                  Average|
|       2.8|                    9.0|                     Good|
|       2.6|                    3.0|            Below Average|
|       3.0|                    9.0|                     Good|
|       2.8|                    5.0|                  Average|
|       2.4|                    7.0|                  Average|
|       3.6|                    8.0|                Excellent|
|       3.6|                    8.0|                Excellent|
|       3.8|                    8.0|                Exc

Calcul du niveau de risque

In [24]:
from pyspark.sql.functions import lit

risk_score = (
    when(col("daily_gaming_hours") >= 8, 3)
    .when(col("daily_gaming_hours") >= 5, 2)
    .when(col("daily_gaming_hours") >= 3, 1)
    .otherwise(0)
    +
    when(col("sleep_hours") < 5, 2)
    .when(col("sleep_hours") < 7, 1)
    .otherwise(0)
    +
    when(col("social_isolation_score") >= 8, 2)
    .when(col("social_isolation_score") >= 5, 1)
    .otherwise(0)
    +
    when(col("withdrawal_symptoms") == True, 1).otherwise(0)
    +
    when(col("loss_of_other_interests") == True, 1).otherwise(0)
    +
    when(col("continued_despite_problems") == True, 2).otherwise(0)
)

form_df = form_df.withColumn("risk_score", risk_score)

form_df = form_df.withColumn(
    "gaming_addiction_risk_level",
    when(col("risk_score") >= 8, "Severe")
    .when(col("risk_score") >= 5, "High")
    .when(col("risk_score") >= 3, "Moderate")
    .otherwise("Low")
)

form_df.select(
    "daily_gaming_hours",
    "sleep_hours",
    "social_isolation_score",
    "withdrawal_symptoms",
    "loss_of_other_interests",
    "continued_despite_problems",
    "risk_score",
    "gaming_addiction_risk_level"
).show()

form_df = form_df.drop("risk_score")

+------------------+-----------+----------------------+-------------------+-----------------------+--------------------------+----------+---------------------------+
|daily_gaming_hours|sleep_hours|social_isolation_score|withdrawal_symptoms|loss_of_other_interests|continued_despite_problems|risk_score|gaming_addiction_risk_level|
+------------------+-----------+----------------------+-------------------+-----------------------+--------------------------+----------+---------------------------+
|               8.0|        7.0|                   2.0|               true|                   true|                      true|         7|                       High|
|              14.0|        4.0|                   4.0|              false|                   true|                      true|         8|                     Severe|
|               2.0|        6.0|                   4.0|               true|                  false|                      true|         4|                   Moderate|
|   

Génération des id de nos nouvelles entrées

In [25]:
from pyspark.sql.functions import regexp_extract, max as spark_max, row_number, format_string
from pyspark.sql.window import Window

max_id = (
    base_df
    .select(regexp_extract(col("record_id"), r"GD(\d+)", 1).cast("int").alias("id_num"))
    .agg(spark_max("id_num").alias("max_id"))
    .collect()[0]["max_id"]
)

print("Dernier ID existant :", max_id)

window = Window.orderBy(lit(1))

form_df = (
    form_df
    .withColumn("row_num", row_number().over(window))
    .withColumn("record_id", format_string("GD%04d", col("row_num") + lit(max_id)))
    .drop("row_num")
)

form_df.select("record_id").show()

Dernier ID existant : 1000
+---------+
|record_id|
+---------+
|   GD1001|
|   GD1002|
|   GD1003|
|   GD1004|
|   GD1005|
|   GD1006|
|   GD1007|
|   GD1008|
|   GD1009|
|   GD1010|
|   GD1011|
|   GD1012|
|   GD1013|
|   GD1014|
|   GD1015|
|   GD1016|
|   GD1017|
|   GD1018|
|   GD1019|
+---------+



26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


mise dans le bon ordre des colonnes

In [26]:
form_df = form_df.select(base_df.columns)

print("Colonnes dataset original :", base_df.columns)
print("Colonnes Google Form transformé :", form_df.columns)

form_df.show(5)
form_df.printSchema()

Colonnes dataset original : ['record_id', 'age', 'gender', 'daily_gaming_hours', 'game_genre', 'primary_game', 'gaming_platform', 'sleep_hours', 'sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'grades_gpa', 'work_productivity_score', 'mood_state', 'mood_swing_frequency', 'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems', 'eye_strain', 'back_neck_pain', 'weight_change_kg', 'exercise_hours_weekly', 'social_isolation_score', 'face_to_face_social_hours_weekly', 'monthly_game_spending_usd', 'years_gaming', 'gaming_addiction_risk_level']
Colonnes Google Form transformé : ['record_id', 'age', 'gender', 'daily_gaming_hours', 'game_genre', 'primary_game', 'gaming_platform', 'sleep_hours', 'sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'grades_gpa', 'work_productivity_score', 'mood_state', 'mood_swing_frequency', 'withdrawal_symptoms', 'loss_of_other_interests', 'continued_despite_problems', 'eye_strain', 'back

26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+----+------+------------------+----------+---------------+---------------+-----------+-------------+--------------------------+-------------------------+----------+-----------------------+----------+--------------------+-------------------+-----------------------+--------------------------+----------+--------------+----------------+---------------------+----------------------+--------------------------------+-------------------------+------------+---------------------------+
|record_id| age|gender|daily_gaming_hours|game_genre|   primary_game|gaming_platform|sleep_hours|sleep_quality|sleep_disruption_frequency|academic_work_performance|grades_gpa|work_productivity_score|mood_state|mood_swing_frequency|withdrawal_symptoms|loss_of_other_interests|continued_despite_problems|eye_strain|back_neck_pain|weight_change_kg|exercise_hours_weekly|social_isolation_score|face_to_face_social_hours_weekly|monthly_game_spending_usd|years_gaming|gaming_addiction_risk_level|
+---------+----+--

Fusion et écriture dans HDFS

In [27]:
merged_df = base_df.unionByName(form_df)

print("Lignes dataset original :", base_df.count())
print("Nouvelles réponses Google Form :", form_df.count())
print("Total après fusion :", merged_df.count())

output_path = "hdfs://localhost:9000/projet/gaming_mental_health/output/merged_dataset"

(
    merged_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_path)
)

print("Dataset fusionné écrit dans :", output_path)

Lignes dataset original : 1000
Nouvelles réponses Google Form : 19
Total après fusion : 1019


26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/02 12:04:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Dataset fusionné écrit dans : hdfs://localhost:9000/projet/gaming_mental_health/output/merged_dataset
